In [20]:
# etapa_1_extracao.py
# Objetivo: ler as tabelas de vendas do banco relacional e preparar DataFrames

import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

# Conexão com o banco fonte
engine = create_engine(os.environ["SOURCE_DATABASE_URL"])



In [21]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

engine_rs = create_engine(
    os.environ["REDSHIFT_DATABASE_URL"],
    connect_args={"sslmode": os.getenv("REDSHIFT_SSLMODE", "require")},
)



In [22]:
def add_sk(df, col_name="sk"):
    df.insert(0, col_name, range(1, len(df) + 1))
    return df

In [25]:
datas = pd.date_range("2015-01-01", "2026-12-31")
dim_tempo = pd.DataFrame({
    "data":          datas,
    "ano":           datas.year,
    "mes":           datas.month,
    "trimestre":     datas.quarter,
    "nome_mes":      datas.month_name(),
    "dia_semana":    datas.day_name(),
    "is_fim_semana": datas.weekday >= 5,
})
add_sk(dim_tempo, "sk_tempo")

with engine_rs.begin() as conn:
    conn.execute(text("TRUNCATE TABLE public.dim_tempo"))

dim_tempo.to_sql(
    "dim_tempo",
    engine_rs,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)
print(f"dim_tempo: {len(dim_tempo)} linhas")

dim_tempo: 4383 linhas


In [26]:
df_produto = pd.read_sql("""
    SELECT
        p.id          AS id_produto,
        p.nome        AS nome,
        cat.descricao AS categoria,
        p.valor_venda AS preco_tabela
    FROM vendas.produto p
    JOIN vendas.categoria cat ON cat.id = p.id_categoria
""", engine)

dim_produto = add_sk(df_produto.copy(), "sk_produto")

with engine_rs.begin() as conn:
    conn.execute(text("TRUNCATE TABLE public.dim_produto"))

dim_produto.to_sql(
    "dim_produto",
    engine_rs,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"dim_produto: {len(dim_produto)} linhas")


dim_produto: 233 linhas


In [27]:
from sqlalchemy import create_engine, text

df_cliente = pd.read_sql("""
    SELECT DISTINCT
        pf.id       AS id_pessoa,
        pf.nome     AS nome,
        pf.cpf
    FROM vendas.nota_fiscal nf
    JOIN geral.pessoa_fisica pf ON pf.id = nf.id_cliente
""", engine)

dim_cliente = add_sk(df_cliente.copy(), "sk_cliente")

with engine_rs.begin() as conn:
    conn.execute(text("TRUNCATE TABLE public.dim_cliente"))

dim_cliente.to_sql(
    "dim_cliente",
    engine_rs,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"dim_cliente: {len(dim_cliente)} linhas")


dim_cliente: 14133 linhas


In [28]:
df_vendedor = pd.read_sql("""
    SELECT DISTINCT
        pf.id   AS id_pessoa,
        pf.nome AS nome
    FROM vendas.nota_fiscal nf
    JOIN geral.pessoa_fisica pf ON pf.id = nf.id_vendedor
""", engine)

dim_vendedor = add_sk(df_vendedor.copy(), "sk_vendedor")

with engine_rs.begin() as conn:
    conn.execute(text("TRUNCATE TABLE public.dim_vendedor"))

dim_vendedor.to_sql(
    "dim_vendedor",
    engine_rs,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"dim_vendedor: {len(dim_vendedor)} linhas")


dim_vendedor: 24 linhas


In [29]:
df_fato_raw = pd.read_sql("""
    SELECT
        nf.data_venda::date        AS data_venda,
        nf.id_cliente,
        nf.id_vendedor,
        inf.id_produto,
        nf.id                      AS id_nota,
        inf.quantidade,
        inf.valor_unitario,
        inf.valor_venda_real,
        p.valor_venda                  
    FROM vendas.nota_fiscal nf
    JOIN vendas.item_nota_fiscal inf ON inf.id_nota_fiscal = nf.id
    JOIN vendas.produto p            ON p.id = inf.id_produto
""", engine)

df_fato = df_fato_raw.copy()

df_fato["data_venda"] = pd.to_datetime(df_fato["data_venda"]).dt.date
dim_tempo["data"] = pd.to_datetime(dim_tempo["data"]).dt.date

df_fato = df_fato.merge(
    dim_tempo[["sk_tempo", "data"]].rename(columns={"data": "data_venda"}),
    on="data_venda",
    how="left"
)


df_fato = df_fato.merge(
    dim_produto[["sk_produto", "id_produto"]],
    on="id_produto",
    how="left"
)

df_fato = df_fato.merge(
    dim_cliente[["sk_cliente", "id_pessoa"]].rename(columns={"id_pessoa": "id_cliente"}),
    on="id_cliente",
    how="left"
)

df_fato = df_fato.merge(
    dim_vendedor[["sk_vendedor", "id_pessoa"]].rename(columns={"id_pessoa": "id_vendedor"}),
    on="id_vendedor",
    how="left"
)

df_fato["desconto"] = (
    df_fato["valor_venda"] - df_fato["valor_unitario"]
).round(2)

df_fato["pct_desconto"] = (
    df_fato["desconto"] / df_fato["valor_unitario"] * 100
).round(2)


colunas_fato = [
    "sk_tempo", "sk_produto", "sk_cliente", "sk_vendedor",
    "id_nota",  "quantidade", "valor_unitario", "valor_venda_real",
    "desconto", "pct_desconto",
]

df_fato = df_fato[colunas_fato]

nulos = df_fato[
    ["sk_tempo", "sk_produto", "sk_cliente", "sk_vendedor"]
].isnull().sum()

if nulos.any():
    print("ATENÇÃO — SKs com nulos (checar joins):")
    print(nulos[nulos > 0])

with engine_rs.begin() as conn:
    conn.execute(text("TRUNCATE TABLE public.fato_venda"))

df_fato.to_sql(
    "fato_venda",
    engine_rs,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=10000
)

print(f"fato_venda: {len(df_fato)} linhas")


ATENÇÃO — SKs com nulos (checar joins):
sk_cliente    41074
dtype: int64
fato_venda: 342491 linhas


In [30]:
from hdfs import InsecureClient
import os

def upload_to_hdfs(local_folder, hdfs_url, hdfs_user, hdfs_dest_path):
    # Conecta ao HDFS
    client = InsecureClient(hdfs_url, user=hdfs_user)

    # Verifica se a pasta existe no HDFS, caso contrário, cria
    if not client.status(hdfs_dest_path, strict=False):
        client.makedirs(hdfs_dest_path)
        print(f"Pasta criada no HDFS: {hdfs_dest_path}")

    # Lê os arquivos no diretório local
    for file_name in os.listdir(local_folder):
        file_path = os.path.join(local_folder, file_name)

        # Verifica se é um arquivo (ignora pastas)
        if os.path.isfile(file_path):
            # Define o caminho completo no HDFS
            hdfs_file_path = f"{hdfs_dest_path}/{file_name}"
            print(f"Fazendo upload de {file_name} para {hdfs_file_path}...")

            with open(file_path, "rb") as file_data:
                client.write(hdfs_file_path, file_data, overwrite=True)

            print(f"Arquivo {file_name} salvo no HDFS.")

In [31]:
hdfs_url = "http://hadoop:9870"
hdfs_user = "root"
hdfs_dest_path = "/vendas/fato_vendas/"

In [32]:
import os
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

local_parquet_folder = "lake/fato_venda"

query = """
SELECT ano, mes, fv.quantidade, fv.valor_unitario, fv.valor_venda_real
FROM public.fato_venda fv
JOIN public.dim_tempo dt ON dt.sk_tempo = fv.sk_tempo
"""

df_fato = pd.read_sql(query, engine_rs)

if os.path.exists(local_parquet_folder):
    shutil.rmtree(local_parquet_folder)

table = pa.Table.from_pandas(df_fato, preserve_index=False)

pq.write_to_dataset(
    table,
    root_path=local_parquet_folder,
    partition_cols=["ano", "mes"],
    compression="snappy"
)

for root, _, files in os.walk(local_parquet_folder):
    parquet_files = [file for file in files if file.endswith(".parquet")]
    if not parquet_files:
        continue

    relative_path = os.path.relpath(root, local_parquet_folder).replace("\\", "/")
    hdfs_partition_path = hdfs_dest_path.rstrip("/")
    if relative_path != ".":
        hdfs_partition_path = f"{hdfs_partition_path}/{relative_path}"

    upload_to_hdfs(
        local_folder=root,
        hdfs_url=hdfs_url,
        hdfs_user=hdfs_user,
        hdfs_dest_path=hdfs_partition_path
    )

print(f"Parquet salvo no HDFS em {hdfs_dest_path} com {len(df_fato)} linhas.")


Fazendo upload de 5a189b07fc4241ce8cb2130e78216e33-0.parquet para /vendas/fato_vendas/ano=2016/mes=1/5a189b07fc4241ce8cb2130e78216e33-0.parquet...
Arquivo 5a189b07fc4241ce8cb2130e78216e33-0.parquet salvo no HDFS.
Pasta criada no HDFS: /vendas/fato_vendas/ano=2016/mes=7
Fazendo upload de 5a189b07fc4241ce8cb2130e78216e33-0.parquet para /vendas/fato_vendas/ano=2016/mes=7/5a189b07fc4241ce8cb2130e78216e33-0.parquet...
Arquivo 5a189b07fc4241ce8cb2130e78216e33-0.parquet salvo no HDFS.
Pasta criada no HDFS: /vendas/fato_vendas/ano=2016/mes=6
Fazendo upload de 5a189b07fc4241ce8cb2130e78216e33-0.parquet para /vendas/fato_vendas/ano=2016/mes=6/5a189b07fc4241ce8cb2130e78216e33-0.parquet...
Arquivo 5a189b07fc4241ce8cb2130e78216e33-0.parquet salvo no HDFS.
Pasta criada no HDFS: /vendas/fato_vendas/ano=2016/mes=10
Fazendo upload de 5a189b07fc4241ce8cb2130e78216e33-0.parquet para /vendas/fato_vendas/ano=2016/mes=10/5a189b07fc4241ce8cb2130e78216e33-0.parquet...
Arquivo 5a189b07fc4241ce8cb2130e78216e33-

In [34]:
from io import BytesIO
from hdfs import InsecureClient
import pyarrow.parquet as pq
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
engine_dm = create_engine(os.environ["DASHBOARD_DATABASE_URL"])

anos = [2024, 2025, 2026]
client = InsecureClient(hdfs_url, user=hdfs_user)

dfs = []

for ano in anos:
    ano_path = f"{hdfs_dest_path.rstrip('/')}/ano={ano}"

    if not client.status(ano_path, strict=False):
        print(f"Ano {ano} n?o encontrado no HDFS: {ano_path}")
        continue

    meses = client.list(ano_path, status=True)

    for mes_nome, mes_status in meses:
        if mes_status["type"] != "DIRECTORY" or not mes_nome.startswith("mes="):
            continue

        mes_path = f"{ano_path}/{mes_nome}"
        arquivos = client.list(mes_path, status=True)

        for arquivo_nome, arquivo_status in arquivos:
            if arquivo_status["type"] != "FILE" or not arquivo_nome.endswith(".parquet"):
                continue

            parquet_path = f"{mes_path}/{arquivo_nome}"
            print(f"Lendo {parquet_path}")

            with client.read(parquet_path) as reader:
                parquet_bytes = BytesIO(reader.read())

            df_part = pq.read_table(parquet_bytes).to_pandas()
            df_part["ano"] = ano
            df_part["mes"] = int(mes_nome.split("=")[1])
            dfs.append(df_part)

if not dfs:
    raise ValueError("Nenhum arquivo Parquet encontrado no HDFS para os anos informados.")

df_vendas_hdfs = pd.concat(dfs, ignore_index=True)

vendas_ano_mes = (
    df_vendas_hdfs
    .groupby(["ano", "mes"], as_index=False)
    .agg(
        qtde_vendida=("quantidade", "sum"),
        valor_total_real=("valor_venda_real", "sum"),
        valor_total_esperado=("valor_unitario", "sum"),
    )
)

vendas_ano_mes["qtde_vendida"] = vendas_ano_mes["qtde_vendida"].astype(int)
vendas_ano_mes["valor_total_real"] = vendas_ano_mes["valor_total_real"].round(2)
vendas_ano_mes["valor_total_esperado"] = vendas_ano_mes["valor_total_esperado"].round(2)

with engine_dm.begin() as conn:
    conn.execute(text("TRUNCATE TABLE public.vendas_ano_mes_jaime"))

vendas_ano_mes.to_sql(
    "vendas_ano_mes_jaime",
    engine_dm,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"vendas_ano_mes: {len(vendas_ano_mes)} linhas salvas")
display(vendas_ano_mes.sort_values(["ano", "mes"]))


Lendo /vendas/fato_vendas/ano=2024/mes=1/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=10/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=11/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=12/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=2/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=3/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=4/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=5/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=6/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=7/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=8/5a189b07fc4241ce8cb2130e78216e33-0.parquet
Lendo /vendas/fato_vendas/ano=2024/mes=9/5a189b07fc4241ce8cb2130e78216e33

,ano,mes,qtde_vendida,valor_total_real,valor_total_esperado
0,2024,1,8705,8158182.10,3346387.24
1,2024,2,8372,8148625.43,3253611.43
2,2024,3,9070,8761825.25,3503264.82
3,2024,4,8774,8612383.91,3394586.97
4,2024,5,8888,8663690.58,3423988.92
5,2024,6,8504,8438650.02,3365852.64
6,2024,7,8920,8322964.43,3366438.35
7,2024,8,8793,8865346.48,3486271.07
8,2024,9,8559,8101188.37,3220528.11
9,2024,10,8940,8779386.36,3563257.87
